In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

# Check if we are currently inside the 'notebooks' folder
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

In [ ]:
from src.config import SimConfig
from src.utils.data_processing import load_and_cache_entire_fleet
from src.utils.evaluation import VoyageBenchmarker

from src.plants.fc_only_plant import FuelCellOnlyPlant
from src.solvers.sdp_baseline import BaselineSDPSolver
from src.controllers.constant import ConstantControl
from src.controllers.threshold import ThresholdControl
from src.controllers.stochastic import StochasticControl
from src.utils.plotting import plot_dynamic_history, plot_cost_comparison, plot_benchmarker_results

config = SimConfig()
fleet_data = load_and_cache_entire_fleet(config)

# Initialize benchmarker, excluding corrupted or unrepresentative data
exclude_days = [] # Adjust this list as needed based on data quality
benchmarker = VoyageBenchmarker(fleet_data, config, exclude_days)

In [ ]:
# We wrap instantiations in functions so the benchmarker can spin up fresh 
# plants for every cross-validation day, preventing state-bleeding.

def build_constant(cfg, mc, horizon):
    return ConstantControl(cfg), FuelCellOnlyPlant(cfg)

def build_threshold(cfg, mc, horizon):
    return ThresholdControl(cfg, horizon, sigma=0.5), FuelCellOnlyPlant(cfg)

def build_sdp(cfg, mc, horizon):
    solver = BaselineSDPSolver(cfg, mc)
    policy = solver.compute_policy_matrix(horizon)
    ctrl = StochasticControl(mc['levels'], cfg.n_vals, policy)
    return ctrl, FuelCellOnlyPlant(cfg)

approaches = {
    "Constant Baseline": build_constant,
    "Threshold Heuristic": build_threshold,
    "Full SDP": build_sdp
}

In [ ]:
print("--- APPROACH A: HAND-PICKED EVALUATION ---")
# Manually choose training block and test validation target
train_days = [4, 5, 6, 7, 8, 9, 10]
test_day = 14
df_manual = benchmarker.compare_approaches(approaches, train_days, test_day)
display(df_manual)


In [ ]:
print("\n--- APPROACH B: LEAVE-ONE-OUT (Full SDP) ---")

# Systematically validate against every single viable day in the dataset
df_loo_sdp = benchmarker.run_leave_one_out(approaches["Full SDP"])
display(df_loo_sdp)

df_loo_thresh = benchmarker.run_leave_one_out(approaches["Threshold Heuristic"])
display(df_loo_thresh)


# Visualize the day-to-day volatility
plot_benchmarker_results(df_loo_sdp, title="Leave-One-Out Cross Validation (SDP)", plot_type='bar')
plot_benchmarker_results(df_loo_thresh, title="Leave-One-Out Cross Validation (Threshold Heuristic)", plot_type='bar')

In [ ]:

print("\n--- APPROACH C: FORWARD CHAINING (Full SDP) ---")
# Evaluate how the policy improves as the agent gathers chronological data
df_for_sdp = benchmarker.run_forward_chaining(approaches["Full SDP"])
display(df_for_sdp)

df_for_thresh = benchmarker.run_forward_chaining(approaches["Threshold Heuristic"])
display(df_for_thresh)


# Visualize the day-to-day volatility
plot_benchmarker_results(df_for_sdp, title="Forward Chaining Learning Curve (SDP)", plot_type='line')
plot_benchmarker_results(df_for_thresh, title="Forward Chaining Learning Curve (Threshold Heuristic)", plot_type='line')